# Stage 6 — Balanced fuzzy rule evaluation

In [ ]:
#@title Shared MFAR paths and stage initialization
from pathlib import Path
from datetime import datetime, timezone
import os, sys, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

# Locate the code repository only; all simulation I/O paths are resolved by
# src.mfar_paths through MFAR_GDRIVE_ROOT or the mounted/synchronized Drive.
_code_candidates = [Path.cwd(), Path.cwd().parent]
if os.environ.get("MFAR_CODE_ROOT"):
    _code_candidates.insert(0, Path(os.environ["MFAR_CODE_ROOT"]))
_code_candidates.extend([
    Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline"),
    Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline/MFAR_Modular_Colab_Pipeline"),
])
for _candidate in _code_candidates:
    if (_candidate / "src" / "mfar_paths.py").is_file():
        sys.path.insert(0, str(_candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "Modul src/mfar_paths.py tidak ditemukan. Jalankan notebook dari repository "
        "atau tetapkan MFAR_CODE_ROOT ke folder repository."
    )

from src.mfar_paths import (
    AIS_RAW_PATH, VEHICLE_ARRIVAL_PATH, DATA_RAW_DIR, CONFIG_DIR,
    STAGE_OUTPUT_DIR, STAGE_01_DIR, STAGE_02_DIR, STAGE_03_DIR,
    STAGE_04_DIR, STAGE_05_DIR, STAGE_06_DIR, STAGE_07_DIR,
    validate_csv_input, validate_raw_inputs, validate_writable_directory,
    write_execution_metadata,
)

_MFAR_STARTED_AT = datetime.now(timezone.utc)

NOTEBOOK_NAME = "06_Rule_Evaluation.ipynb"
RAW, CFG, STAGE = DATA_RAW_DIR, CONFIG_DIR, STAGE_OUTPUT_DIR
validate_writable_directory(STAGE_06_DIR, NOTEBOOK_NAME, 6)
print("Input stage:", STAGE_05_DIR)
print("Output folder:", STAGE_06_DIR)


In [ ]:
df=validate_csv_input(STAGE_05_DIR/"05_fuzzy_memberships.csv", ["simulation_time","origin","selected_action"] if False else ["simulation_time","origin","mu_origin_queue_low","mu_berth_available","mu_berth_unavailable"], NOTEBOOK_NAME, 6)
def FAND(*v): return float(min(float(x) for x in v))
def FOR(*v): return float(max(float(x) for x in v))

RULE_TEXT={
"R01":"IF origin queue LOW AND berth AVAILABLE THEN NO_INTERVENTION",
"R02":"IF destination queue LOW AND berth AVAILABLE THEN MAINTAIN_SPEED",
"R03":"IF origin queue MEDIUM OR HIGH AND berth AVAILABLE THEN DEPART_NOW",
"R04":"IF berth UNAVAILABLE AND wait MEDIUM OR LONG THEN HOLD_DEPARTURE",
"R05":"IF berth UNAVAILABLE AND wait MEDIUM OR LONG THEN REDUCE_SPEED",
"R06":"IF origin queue HIGH AND service gap MEDIUM OR LONG THEN INCREASE_SERVICE_PRIORITY",
"R07":"IF destination queue HIGH OR CRITICAL AND wait MEDIUM OR LONG THEN RESCHEDULE_HEADWAY",
"R08":"IF origin queue CRITICAL AND service gap LONG AND capacity shortfall HIGH THEN ADD_VESSEL",
"R09":"IF confidence LOW AND operational risk exists THEN ALERT_OPERATOR",
"R10":"IF origin queue CRITICAL AND berth UNAVAILABLE THEN ALERT_OPERATOR"
}

In [ ]:
def infer(r):
    rules={
     "R01":(FAND(r.mu_origin_queue_low,r.mu_berth_available),"NO_INTERVENTION"),
     "R02":(FAND(r.mu_destination_queue_low,r.mu_berth_available),"MAINTAIN_SPEED"),
     "R03":(FAND(FOR(r.mu_origin_queue_medium,r.mu_origin_queue_high),
                  r.mu_berth_available),"DEPART_NOW"),
     "R04":(FAND(r.mu_berth_unavailable,FOR(r.mu_wait_medium,r.mu_wait_long)),
            "HOLD_DEPARTURE"),
     "R05":(FAND(r.mu_berth_unavailable,FOR(r.mu_wait_medium,r.mu_wait_long)),
            "REDUCE_SPEED"),
     "R06":(FAND(r.mu_origin_queue_high,
                  FOR(r.mu_service_gap_medium,r.mu_service_gap_long)),
            "INCREASE_SERVICE_PRIORITY"),
     "R07":(FAND(FOR(r.mu_destination_queue_high,r.mu_destination_queue_critical),
                  FOR(r.mu_wait_medium,r.mu_wait_long)),
            "RESCHEDULE_HEADWAY"),
     "R08":(FAND(r.mu_origin_queue_critical,r.mu_service_gap_long,
                  r.mu_capacity_shortfall_high),"ADD_VESSEL"),
     "R09":(FAND(r.mu_confidence_low,
                  FOR(r.mu_origin_queue_high,r.mu_origin_queue_critical,
                      r.mu_berth_unavailable)),"ALERT_OPERATOR"),
     "R10":(FAND(r.mu_origin_queue_critical,r.mu_berth_unavailable),
            "ALERT_OPERATOR")
    }
    actions={}
    for rid,(s,a) in rules.items():
        actions[a]=max(actions.get(a,0.0),s)
    if max(actions.values() or [0])==0:
        actions["NO_INTERVENTION"]=1.0
    sel=max(actions,key=actions.get)
    ss=actions[sel]
    dominant="|".join(rid for rid,(s,a) in rules.items()
                      if a==sel and abs(s-ss)<1e-12)
    return pd.Series({
      **{f"firing_{rid}":s for rid,(s,a) in rules.items()},
      "selected_action":sel,"selected_rule_strength":ss,
      "dominant_rule":dominant
    })

out=pd.concat([df,df.apply(infer,axis=1)],axis=1)
S6=STAGE/"stage_06"
out.to_csv(S6/"06_rule_evaluation.csv",index=False)
pd.DataFrame({"rule_id":list(RULE_TEXT),"rule_text":list(RULE_TEXT.values())}).to_csv(
 S6/"06_fuzzy_rule_catalog.csv",index=False)
pd.DataFrame([
 {"rule_id":rid,"active_rows":int((out[f"firing_{rid}"]>0).sum()),
  "mean_firing_strength":float(out[f"firing_{rid}"].mean()),
  "max_firing_strength":float(out[f"firing_{rid}"].max())}
 for rid in RULE_TEXT
]).to_csv(S6/"06_rule_firing_summary.csv",index=False)
display(out["selected_action"].value_counts())

## Keluaran interpretatif otomatis\n\nCSV/JSON dipertahankan untuk kontrak data. Sel berikut membuat keluaran yang dapat dibaca dan dieksplorasi tanpa membuka CSV mentah.

In [ ]:
#@title Export readable Stage 6 outputs
from src.mfar_visuals import stage6_rule_outputs
_rule_catalog = pd.DataFrame({
    "rule_id": list(RULE_TEXT),
    "rule_text": list(RULE_TEXT.values())
})
_rule_summary = pd.DataFrame([
    {
        "rule_id": rid,
        "active_rows": int((out[f"firing_{rid}"] > 0).sum()),
        "mean_firing_strength": float(out[f"firing_{rid}"].mean()),
        "max_firing_strength": float(out[f"firing_{rid}"].max()),
    }
    for rid in RULE_TEXT
])
_readable_outputs = stage6_rule_outputs(
    out, _rule_catalog, _rule_summary, STAGE_06_DIR
)
print("Readable Stage 6 outputs:")
for _path in _readable_outputs:
    print("-", _path.name)


In [ ]:
#@title Execution metadata and saved-artifact report
_stage_dir = STAGE_06_DIR
_saved_files = sorted(_stage_dir.glob("06_*"))
write_execution_metadata(
    stage=6, notebook=NOTEBOOK_NAME, started_at=_MFAR_STARTED_AT,
    input_paths=[STAGE_05_DIR/'05_fuzzy_memberships.csv'],
    input_rows={"df": len(df)},
    output_rows={"out": len(out)},
    output_files=_saved_files,
)
